# Clinical Plausibility Constraints

Adversarial attacks on image classifiers can perturb pixel values by tiny amounts. The
change is invisible to humans but fools the model. Tabular health data is fundamentally
different. Every feature has a **real-world meaning** and not all perturbations are
physically or clinically possible.

An attack that succeeds by setting a patient's Glucose to -40 mg/dL or BMI to 0.0 is
**clinically meaningless**. No real patient could submit those values. If we count such
attacks as successes our robustness analysis is inflated.

**This section defines the constraint layer that wraps every attack.**
No adversarial example will be accepted as valid unless it passes through this layer.

### Threat Model Assumption
We assume a **black-box, clinically-constrained adversary**:
- Has access to model outputs only (not model weights or training data)
- Can only submit values a real human patient could plausibly have
- Cannot change categorical features to impossible values
- Cannot violate logical dependencies between features

This reflects the most realistic attack scenario: a patient manipulating their own
self-reported or measured clinical data to game a risk screener.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field, asdict
from typing import Optional, List

In [2]:
DATA_DIR = '../data/processed/'
OUTPUT_DIR = '../output/'

diabetes_df = pd.read_csv(os.path.join(DATA_DIR, 'diabetes_processed.csv'))
heart_df    = pd.read_csv(os.path.join(DATA_DIR, 'heart_processed.csv'))
stroke_df   = pd.read_csv(os.path.join(DATA_DIR, 'stroke_processed.csv'))

print(f'  Diabetes : {diabetes_df.shape}')
print(f'  Heart    : {heart_df.shape}')
print(f'  Stroke   : {stroke_df.shape}')

  Diabetes : (768, 9)
  Heart    : (1025, 14)
  Stroke   : (4254, 12)


---
## Constraint Design Principles

Each feature constraint is defined by

| Property | Meaning |
|---|---|
| `min` / `max` | Hard physiological or clinical bounds. Values outside these are impossible |
| `mutable` | Whether an adversary can change this feature at all |
| `is_categorical` | Whether the feature takes only discrete values |
| `allowed_values` | Exact allowed values for categorical features |
| `logical_deps` | Cross-feature constraints (e.g., Pregnancies only valid for females) |

### Three Types of Constraints

**Physiological bounds**: Based on what a living human can have.
e.g., Glucose cannot be negative. BMI below 10 is incompatible with life.

**Categorical rigidity**: Binary or ordinal features do not perturb
continuously. Gender does not shift from 0.0 to 0.3. It is either 0 or 1.

**Logical dependencies**: Some features are only valid given the state
of another. Pregnancies > 0 is only valid for biological females.

In [4]:
# Feature Constraint Class
@dataclass
class FeatureConstraint:
    """
    Defines the clinical plausibility constraints for a single feature.
    Used to validate adversarial examples before accepting them as valid attacks.
    """
    name: str
    min_val: Optional[float]       # Hard lower bound (None = no lower bound)
    max_val: Optional[float]       # Hard upper bound (None = no upper bound)
    mutable: bool                  # Can an adversary perturb this feature?
    is_categorical: bool           # Does it take only discrete values?
    allowed_values: Optional[List] # If categorical, what values are allowed?
    clinical_source: str           # Where this bound comes from
    notes: str = ''                # Any extra reasoning

    def validate(self, value) -> bool:
        """Returns True if a value is clinically plausible."""
        if self.is_categorical and self.allowed_values is not None:
            return value in self.allowed_values
        if self.min_val is not None and value < self.min_val:
            return False
        if self.max_val is not None and value > self.max_val:
            return False
        return True

    def clip(self, value):
        """Clips a continuous value to the valid range."""
        if self.is_categorical:
            raise ValueError(f'{self.name} is categorical — use round/map instead of clip')
        val = value
        if self.min_val is not None:
            val = max(val, self.min_val)
        if self.max_val is not None:
            val = min(val, self.max_val)
        return val

## DIABETES CONSTRAINTS 
Source references used
- WHO Diabetes Diagnosis Criteria (Glucose thresholds)
- CDC BMI Classification Guidelines
- AHA Blood Pressure Guidelines
- Clinical reference ranges for insulin and skin thickness
- PIMA Indians dataset documentation (Lichman, 2013)

In [5]:
DIABETES_CONSTRAINTS = [

    FeatureConstraint(
        name='Pregnancies',
        # Dataset max is 17. Highest recorded medically is 38, but
        # capping at 20 covers all realistic cases and our data max.
        min_val=0, max_val=20,
        mutable=False,  # An adversary cannot fabricate pregnancy history
        is_categorical=False,
        allowed_values=None,
        clinical_source='PIMA dataset documentation; biological female population only',
        notes='Immutable: this is a historical fact the patient cannot alter on a form'
    ),

    FeatureConstraint(
        name='Glucose',
        # Physiological minimum: below ~40 mg/dL causes unconsciousness (severe hypoglycaemia).
        # A conscious patient submitting a form cannot have glucose < 40.
        # Maximum: above 600 mg/dL is hyperosmolar hyperglycaemic state i.e. patient would be
        # hospitalised, not filling in a risk form. Our data max is 199.
        min_val=40, max_val=400,
        mutable=True,  # KEY: this is the primary attack vector for diabetes evasion
        is_categorical=False,
        allowed_values=None,
        clinical_source='WHO 2006 Diabetes Diagnosis; ADA hypoglycaemia thresholds',
        notes='Primary adversarial target. Underreporting glucose is the most direct evasion.'
    ),

    FeatureConstraint(
        name='BloodPressure',
        # Diastolic BP. Below 40 mmHg = severe shock. Above 140 mmHg = hypertensive crisis.
        # A patient filling in a form will have values in survivable range.
        min_val=40, max_val=140,
        mutable=True,
        is_categorical=False,
        allowed_values=None,
        clinical_source='AHA Blood Pressure Guidelines 2017',
        notes='Diastolic BP. Less likely attack vector than Glucose.'
    ),

    FeatureConstraint(
        name='SkinThickness',
        # Triceps skin fold thickness in mm. Physiological range for adults: 5–60mm.
        # Below 5mm is not clinically observed in living adults.
        min_val=5, max_val=80,
        mutable=True,
        is_categorical=False,
        allowed_values=None,
        clinical_source='Durnin & Womersley 1974 body fat estimation norms',
        notes='Moderate attack vector. Hard to self-report convincingly.'
    ),

    FeatureConstraint(
        name='Insulin',
        # 2-hour serum insulin. Fasting: 2–25 uIU/mL. Post-meal can reach 200+.
        # Our data has values up to 846 (outliers). Cap at 500 as plausible upper bound.
        min_val=2, max_val=500,
        mutable=True,
        is_categorical=False,
        allowed_values=None,
        clinical_source='ADA insulin reference ranges; Tietz Fundamentals of Clinical Chemistry',
        notes='High variance in data. Adversary could plausibly underreport.'
    ),

    FeatureConstraint(
        name='BMI',
        # WHO defines severe underweight as BMI < 16. Below 13 is incompatible with life.
        # Above 80 is extremely rare but documented in severe obesity cases.
        # Our data max is 67.1.
        min_val=13.0, max_val=80.0,
        mutable=True,
        is_categorical=False,
        allowed_values=None,
        clinical_source='WHO BMI Classification 2000; clinical case literature',
        notes='Second most likely attack vector after Glucose. Adversary underreports weight.'
    ),

    FeatureConstraint(
        name='DiabetesPedigreeFunction',
        # DPF is a computed score from family history. Range in literature: 0.078–2.42.
        # Immutable because it is derived from genetic family history — cannot be falsified
        # on a form without access to relatives' records.
        min_val=0.05, max_val=2.50,
        mutable=False,
        is_categorical=False,
        allowed_values=None,
        clinical_source='Smith et al. 1988 (original PIMA study); dataset documentation',
        notes='Immutable: derived from verifiable family medical history'
    ),

    FeatureConstraint(
        name='Age',
        # Dataset is restricted to adults 21+. Max in data is 81.
        # Immutable: age is verifiable from identity documents.
        min_val=21, max_val=100,
        mutable=False,
        is_categorical=False,
        allowed_values=None,
        clinical_source='PIMA dataset documentation; dataset age range 21–81',
        notes='Immutable: verifiable from identity documents'
    ),
]